# Geometric agreement of the flexible-head fits

Recompute the flexible-head geometric quantities used in the choice-head ablation from the fitted coordinates. The same quantities for the positive symmetric head are generated by notebook 14.

**Inputs:** the five included coordinate fits in `outputs/models/flexible/Chile_dim2_seed*`, which notebook 04b can regenerate from the included pairwise observations. **Outputs:** Pearson correlation matrices for participant and proposal distances, a signed PC1 Spearman matrix, and their descriptive ranges in `outputs/ablation/flexible/`.

Run from the repository root or `notebooks/` in a fresh kernel. This notebook needs no TensorFlow and does not train models. It computes all distinct entity pairs in float64, without pair sampling. `BLOCK_SIZE` controls temporary memory, not the statistical sample.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, pdist
from scipy.linalg import svd
from scipy.stats import spearmanr
from sklearn.utils.extmath import svd_flip
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "outputs/models/flexible"
OUTPUT = ROOT / "outputs/ablation/flexible"
SEEDS = [101, 201, 301, 401, 501]
FIT_NAMES = [f"S{i}" for i in range(len(SEEDS))]
BLOCK_SIZE = 1024
assert len(SEEDS) >= 2 and len(SEEDS) == len(set(SEEDS))
assert isinstance(BLOCK_SIZE, int) and BLOCK_SIZE > 0
OUTPUT.mkdir(parents=True, exist_ok=True)

## Match entities across fits

Identifiers must be unique and describe exactly the same people and proposals in every fit. Sorting by identifier makes the correspondence explicit. The coordinate values are not reflected or rotated before calculating distances.

In [2]:
arrays = {}
for entity, filename, key in [("participants", "coordsU.csv", "uuid"), ("proposals", "coordsP.csv", "id")]:
    frames = []
    for seed in SEEDS:
        file = DATA / f"Chile_dim2_seed{seed}" / filename
        if not file.is_file():
            raise FileNotFoundError(f"Missing {file}. Restore the supplied fits or regenerate them in notebook 04b.")
        frame = pd.read_csv(file, dtype={key: str} if key == "uuid" else None)
        assert frame[key].is_unique and frame[key].notna().all()
        assert {key, "z1", "z2"}.issubset(frame.columns)
        frame = frame.sort_values(key, kind="mergesort").reset_index(drop=True)
        if frames:
            pd.testing.assert_series_equal(frame[key], frames[0][key])
        assert np.isfinite(frame[["z1", "z2"]]).all().all()
        frames.append(frame)
    arrays[entity] = [frame[["z1", "z2"]].to_numpy(dtype=np.float64) for frame in frames]
    display(pd.DataFrame({"entity": entity, "seed": SEEDS,
                           "rows": [len(frame) for frame in frames]}))

,entity,seed,rows
0,participants,101,47574
1,participants,201,47574
2,participants,301,47574
3,participants,401,47574
4,participants,501,47574


,entity,seed,rows
0,proposals,101,90
1,proposals,201,90
2,proposals,301,90
3,proposals,401,90
4,proposals,501,90


## Exact correlations without storing full distance matrices

Visit each strict upper-triangle pair once. Accumulate centered cross-products using a blockwise mean/covariance update, so temporary storage is proportional to the block size rather than the square of the participant count. A small deterministic example checks the calculation against SciPy's condensed distance vectors.

In [3]:
def distance_correlations(coordinates, block_size=1024):
    coordinates = [np.asarray(c, dtype=np.float64) for c in coordinates]
    if len(coordinates) < 2 or coordinates[0].ndim != 2:
        raise ValueError("At least two aligned coordinate arrays are required.")
    n, k = coordinates[0].shape
    if n < 3 or k < 1 or any(c.shape != (n, k) or not np.isfinite(c).all() for c in coordinates):
        raise ValueError("Coordinate arrays must be finite and have identical nontrivial shapes.")
    if not isinstance(block_size, int) or block_size <= 0:
        raise ValueError("block_size must be a positive integer.")
    count = 0
    mean = np.zeros(len(coordinates), dtype=np.float64)
    centered_products = np.zeros((len(coordinates), len(coordinates)), dtype=np.float64)
    for start_i in range(0, n, block_size):
        stop_i = min(start_i + block_size, n)
        for start_j in range(start_i, n, block_size):
            stop_j = min(start_j + block_size, n)
            distances = []
            for c in coordinates:
                block = cdist(c[start_i:stop_i], c[start_j:stop_j], metric="euclidean")
                values = block[np.triu_indices(stop_i - start_i, k=1)] if start_i == start_j else block.ravel()
                distances.append(values)
            values = np.stack(distances)
            m = values.shape[1]
            if m == 0:
                continue
            block_mean = values.mean(axis=1)
            values -= block_mean[:, None]
            delta = block_mean - mean
            total = count + m
            centered_products += values @ values.T + np.outer(delta, delta) * (count * m / total)
            mean += delta * (m / total)
            count = total
    if count != n * (n - 1) // 2:
        raise AssertionError("The block traversal did not cover every distinct pair exactly once.")
    variance = np.diag(centered_products)
    if np.any(variance <= 0):
        raise ValueError("A distance vector has zero variance, so its correlation is undefined.")
    result = centered_products / np.sqrt(np.outer(variance, variance))
    result = np.clip((result + result.T) / 2, -1, 1)
    np.fill_diagonal(result, 1)
    return result

rng = np.random.default_rng(317)
example = [rng.normal(size=(31, 2)) for _ in range(5)]
reference = np.corrcoef(np.stack([pdist(c) for c in example]))
for block_size in [1, 7, 31, 64]:
    np.testing.assert_allclose(distance_correlations(example, block_size), reference, rtol=0, atol=2e-12)
print("Blockwise distances agree with the full condensed-vector calculation.")

Blockwise distances agree with the full condensed-vector calculation.


## Participant and proposal distances

Both outputs use signed Pearson correlations. Their ranges describe variation across initializations, not sampling uncertainty.

In [4]:
matrices = {}
for entity in ["proposals", "participants"]:
    print(f"Computing all distinct {entity} pairs...")
    matrix = distance_correlations(arrays[entity], BLOCK_SIZE)
    matrices[entity] = matrix
    frame = pd.DataFrame(matrix, index=FIT_NAMES, columns=FIT_NAMES)
    frame.index.name = "fit"
    display(frame)
    frame.to_csv(OUTPUT / f"geometry_{entity}_pearson.csv")

Computing all distinct proposals pairs...


,S0,S1,S2,S3,S4
fit,,,,,
S0,1.000000,0.046111,0.172522,-0.122343,0.427980
S1,0.046111,1.000000,0.236341,0.153150,0.087961
S2,0.172522,0.236341,1.000000,0.009546,0.128416
S3,-0.122343,0.153150,0.009546,1.000000,-0.119681
S4,0.427980,0.087961,0.128416,-0.119681,1.000000


Computing all distinct participants pairs...


,S0,S1,S2,S3,S4
fit,,,,,
S0,1.000000,0.479255,0.300583,0.434988,0.629164
S1,0.479255,1.000000,0.170283,0.301190,0.456886
S2,0.300583,0.170283,1.000000,0.171135,0.226990
S3,0.434988,0.301190,0.171135,1.000000,0.399202
S4,0.629164,0.456886,0.226990,0.399202,1.000000


## PC1 ordering and summary

PC1 is fitted to all participants in each solution. Its sign is arbitrary. Save the signed matrix for exact correspondence with notebook 18, which takes the absolute value for every fit pair before summarizing the range.

In [5]:
pc1 = []
for c in arrays["participants"]:
    centered = c - c.mean(axis=0)
    u, singular, vt = svd(centered, full_matrices=False)
    if singular[0] <= 0:
        raise ValueError("PC1 is undefined for constant coordinates.")
    u, vt = svd_flip(u, vt)
    pc1.append(u[:, 0] * singular[0])
rank_matrix = np.eye(len(SEEDS))
for i in range(len(SEEDS)):
    for j in range(i + 1, len(SEEDS)):
        rank_matrix[i, j] = rank_matrix[j, i] = spearmanr(pc1[i], pc1[j]).statistic
assert np.isfinite(rank_matrix).all()
matrices["pc1"] = rank_matrix
rank_frame = pd.DataFrame(rank_matrix, index=FIT_NAMES, columns=FIT_NAMES)
rank_frame.index.name = "fit"
display(rank_frame)
rank_frame.to_csv(OUTPUT / "geometry_pca_spearman.csv")
upper = np.triu_indices(len(SEEDS), k=1)
rows = []
for quantity, matrix in matrices.items():
    values = matrix[upper]
    if quantity == "pc1":
        values = np.abs(values)
    rows.append({"quantity": quantity, "fit_pairs": len(values),
                 "minimum": values.min(), "maximum": values.max()})
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(OUTPUT / "geometry_summary.csv", index=False)
fit_seeds = pd.DataFrame({"fit": FIT_NAMES, "seed": SEEDS})
display(fit_seeds)
fit_seeds.to_csv(OUTPUT / "fit_seeds.csv", index=False)

,S0,S1,S2,S3,S4
fit,,,,,
S0,1.000000,-0.625262,0.225152,0.362574,-0.833053
S1,-0.625262,1.000000,-0.010037,-0.225503,0.632871
S2,0.225152,-0.010037,1.000000,0.121586,-0.141044
S3,0.362574,-0.225503,0.121586,1.000000,-0.327313
S4,-0.833053,0.632871,-0.141044,-0.327313,1.000000


,quantity,fit_pairs,minimum,maximum
0,proposals,10,-0.122343,0.427980
1,participants,10,0.170283,0.629164
2,pc1,10,0.010037,0.833053


,fit,seed
0,S0,101
1,S1,201
2,S2,301
3,S3,401
4,S4,501
